# 05 — Edge-IIoTset Model Evaluation

**Goal:** Investigate the nearly perfect validation scores and possible port shortcuts.  
**Input:** Frozen Core16 training/validation files and five saved model pipelines.  
**Output:** Validation errors, attack-type results, and a controlled LightGBM port ablation.  
**Rule:** Use **all** frozen train/validation records; **never open the test split**. This is a *packet/protocol* dataset, not verified flow data.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
import json, hashlib, inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
from lightgbm import LGBMClassifier

ROOT = Path(r'D:\J26-CS-336-Network-Flow-V2\network_flow_v2')
SPLITS = ROOT/'data/splits/edge_iiotset'
TRAIN = SPLITS/'edge_iiotset_core16_train_v3.csv.gz'
VALID = SPLITS/'edge_iiotset_core16_validation_v3.csv.gz'
SPLIT_REPORT = ROOT/'reports/audits/edge_iiotset_v3_split_report.json'
MODEL_DIR = ROOT/'models/edge_iiotset_v3'
AUDITS = ROOT/'reports/audits'
METRICS = ROOT/'reports/metrics'
FIGURES = ROOT/'reports/figures'
MODEL_NAMES = ['decision_tree','random_forest','extra_trees','lightgbm','adaboost']
PORTS = ['tcp.srcport','tcp.dstport']
SEED = 42
MISSING = {'','-','?','na','n/a','nan','none','null','missing','unknown','<missing>'}
for f in (TRAIN, VALID, SPLIT_REPORT):
    if not f.is_file():
        raise FileNotFoundError(f'Notebook 03 input is missing: {f}')
report=json.loads(SPLIT_REPORT.read_text(encoding='utf-8'))
FEATURES=list(report['selected_features'])
CAT=list(report['categorical_features'])
NUM=[f for f in FEATURES if f not in CAT]
if len(FEATURES)!=16 or len(CAT)!=2 or not set(PORTS).issubset(NUM):
    raise ValueError('Core16 feature definitions changed; review Notebook 04 first.')
expected = {p:int(report['counts'][p]['rows']) for p in ('train','validation','test')}
if sum(expected.values()) != 157800 or report['normalized_feature_group_overlap'] != 0:
    raise ValueError('Frozen split integrity check failed.')
if report.get('test_scored') is True:
    raise ValueError('A previous experiment scored the test split; investigate protocol.')
for d in (AUDITS,METRICS,FIGURES,MODEL_DIR): d.mkdir(parents=True,exist_ok=True)
print('Edge-IIoTset Core16 | Training:',expected['train'],'| Validation:',expected['validation'])
print('TEST NOT READ. Original train/validation/test files are never overwritten.')

## 1. Load and verify saved experiment

Check row counts, column order and labels. Recalculate every saved model's confusion matrix.

In [ ]:
columns=['csv_row_index_0based']+FEATURES+['Attack_label','Attack_type']
train=pd.read_csv(TRAIN,dtype=str,keep_default_na=False,low_memory=False)
valid=pd.read_csv(VALID,dtype=str,keep_default_na=False,low_memory=False)
for name, df in [('train',train),('validation',valid)]:
    if df.columns.tolist()!=columns or len(df)!=expected[name]:
        raise ValueError(f'{name} columns or counts do not match frozen manifest.')
    if df['csv_row_index_0based'].duplicated().any() or not df['Attack_label'].isin(['0','1']).all():
        raise ValueError(f'{name} has duplicate row IDs or unexpected class labels.')
    r=report['counts'][name]
    if (int((df['Attack_label']=='0').sum())!=int(r['normal']) or
        int((df['Attack_label']=='1').sum())!=int(r['attack'])):
        raise ValueError(f'{name} label counts differ from split report.')
if set(train['csv_row_index_0based']) & set(valid['csv_row_index_0based']):
    raise ValueError('Train/validation record identifiers overlap.')
labels=valid['Attack_label'].astype(np.int8).to_numpy()
types=valid['Attack_type'].astype(str).to_numpy()
print('Full records verified:',len(train),'+',len(valid),'| Validation normal:',sum(labels==0),'attack:',sum(labels==1))
type_counts=pd.DataFrame({'train':train['Attack_type'].value_counts(),
                          'validation':valid['Attack_type'].value_counts()}).fillna(0).astype(int)
missing_types=type_counts.index[(type_counts['train']>0)&(type_counts['validation']==0)].tolist()
print('Categories absent from validation:',missing_types or 'None')
display(type_counts)

## 2. Verify the five saved models

A saved model must reproduce Notebook 04's validation confusion matrix before we analyze it.

In [ ]:
def clean(frame, columns_to_keep):
    out=frame[columns_to_keep].copy()
    for field in columns_to_keep:
        v=out[field].astype(str).str.strip()
        if field in CAT:
            out[field]=v.mask(v.str.casefold().isin(MISSING),'__MISSING__')
        else:
            numbers=pd.to_numeric(v.mask(v.str.casefold().isin(MISSING)),errors='coerce')
            out[field]=numbers.replace([np.inf,-np.inf],np.nan).astype(np.float32)
    return out

X_valid=clean(valid,FEATURES)
predictions={}
records=[]
for name in MODEL_NAMES:
    model_file=MODEL_DIR/f'{name}_core16_v3.joblib'
    json_file=AUDITS/f'edge_iiotset_v3_{name}_training.json'
    if not model_file.is_file() or not json_file.is_file():
        raise FileNotFoundError(f'Missing saved Notebook 04 artifact for {name}.')
    old=json.loads(json_file.read_text(encoding='utf-8'))
    if old.get('frozen_manifest_sha256')!=report.get('manifest_sha256'):
        raise ValueError(f'{name}: model report uses a different split.')
    model=joblib.load(model_file)  # Load only your own Notebook 04 artifacts.
    pred=np.asarray(model.predict(X_valid)).astype(np.int8)
    tn,fp,fn,tp=map(int,confusion_matrix(labels,pred,labels=[0,1]).ravel())
    original=old['metrics']
    if [tn,fp,fn,tp]!=[int(original[k]) for k in ('tn','fp','fn','tp')]:
        raise ValueError(f'{name}: predictions changed from Notebook 04: {[tn,fp,fn,tp]}')
    predictions[name]=pred
    records.append({'model':name,'accuracy_pct':100*accuracy_score(labels,pred),
                    'attack_f1_pct':100*f1_score(labels,pred),
                    'false_positive_pct':100*fp/max(1,tn+fp),
                    'false_alarms':fp,'missed_attacks':fn})
metrics=pd.DataFrame(records)
display(metrics.round(3))
metrics.to_csv(METRICS/'edge_iiotset_v3_model_evaluation.csv',index=False)
print('All five models matched the saved training reports.')

## 3. Detection by attack category

Only calculate recall for categories actually present in validation. MITM has no validation records.

In [ ]:
attack_rows=[]
for name in MODEL_NAMES:
    pred=predictions[name]
    for typ in sorted(set(types)):
        ids=(types==typ)&(labels==1)
        n=int(ids.sum())
        if n==0: continue
        detected=int(np.sum(pred[ids]==1))
        attack_rows.append({'model':name,'attack_type':typ,'attack_records':n,
            'detected':detected,'missed':n-detected,'recall_pct':100*detected/n})
attack_df=pd.DataFrame(attack_rows)
attack_df.to_csv(METRICS/'edge_iiotset_v3_attack_type_recall.csv',index=False)
display(attack_df[attack_df['model'].isin(['lightgbm','decision_tree','random_forest'])]
        .sort_values(['attack_type','model']).round(2))
print('MITM recall is NOT known:',missing_types)

## 4. Check port patterns

Check how well *port values alone* separate labels. This is a possible dataset shortcut, not proof of leakage.

In [ ]:
# Keep integer-looking port values as numbers; diagnostic only.
port_train=clean(train,PORTS)
port_valid=X_valid[PORTS].copy()
for field in PORTS:
    for part,source in [('train',port_train),('validation',port_valid)]:
        values=source[field]
        print(field,part,'missing/invalid:',int(values.isna().sum()),
              '| zero:',int((values==0).sum()))
# Check the 15 most common port pairs on validation; no IP addresses or payloads.
port_table=port_valid.astype('string').fillna('__INVALID__').copy()
port_table['label']=np.where(labels==1,'Attack','Normal')
port_summary=port_table.groupby(PORTS+['label']).size().unstack(fill_value=0)
for col in ['Normal','Attack']:
    if col not in port_summary: port_summary[col]=0
port_summary=port_summary.reset_index()
port_summary['records']=port_summary['Normal']+port_summary['Attack']
port_summary['attack_pct']=100*port_summary['Attack']/port_summary['records']
port_summary=port_summary.sort_values('records',ascending=False)
port_summary.to_csv(METRICS/'edge_iiotset_v3_port_diagnostics.csv',index=False)
display(port_summary.head(15).round(2))

## 5. LightGBM port-ablation experiments

Retrain **two diagnostic LightGBM models on the full training partition**: (1) ports only; (2) 14 features without ports. Compare both against the already-trained Core16 LightGBM. This does **not** replace Notebook 04's models.

In [ ]:
def build_preprocessor(selected):
    nums=[f for f in selected if f not in CAT]
    cats=[f for f in selected if f in CAT]
    ops=[]
    kwargs={'strategy':'median'}
    if 'keep_empty_features' in inspect.signature(SimpleImputer).parameters:
        kwargs['keep_empty_features']=True
    if nums: ops.append(('numeric',SimpleImputer(**kwargs),nums))
    if cats:
        enc={'handle_unknown':'ignore'}
        if 'sparse_output' in inspect.signature(OneHotEncoder).parameters:
            enc['sparse_output']=True
        else: enc['sparse']=True
        ops.append(('category',OneHotEncoder(**enc),cats))
    return ColumnTransformer(ops,sparse_threshold=1.0)

no_ports=[f for f in FEATURES if f not in PORTS]
X_train=clean(train,FEATURES)
y_train=train['Attack_label'].astype(np.int8).to_numpy()
# Core16 training groups with both labels: a model cannot distinguish these rows.
train_sig=pd.util.hash_pandas_object(X_train,index=False)
group_stats=pd.DataFrame({'fingerprint':train_sig.to_numpy(),'label':y_train}).groupby('fingerprint')['label'].agg(['nunique','size'])
mixed=group_stats[group_stats['nunique']>1]
mixed_training_rows=int(mixed['size'].sum())
print('Core16 conflicting-label groups in training:',len(mixed),'| affected training rows:',mixed_training_rows)
ablations=[{'variant':'full_core16','selected_features':len(FEATURES),
            'accuracy_pct':100*accuracy_score(labels,predictions['lightgbm']),
            'attack_f1_pct':100*f1_score(labels,predictions['lightgbm']),
            'false_positive_pct':100*sum((predictions['lightgbm']==1)&(labels==0))/sum(labels==0),
            'missed_attacks':int(sum((predictions['lightgbm']==0)&(labels==1))),
            'false_alarms':int(sum((predictions['lightgbm']==1)&(labels==0))),
            'training_seconds':np.nan,'model_size_mib':(MODEL_DIR/'lightgbm_core16_v3.joblib').stat().st_size/1048576}]
ablation_preds={'full_core16':predictions['lightgbm']}

for variant,selected in [('ports_only',PORTS),('without_ports',no_ports)]:
    model=Pipeline([('preprocessing',build_preprocessor(selected)),
      ('classifier',LGBMClassifier(n_estimators=160,num_leaves=31,learning_rate=.05,
                                min_child_samples=30,random_state=SEED,n_jobs=4,verbosity=-1))])
    t0=perf_counter()
    model.fit(X_train[selected],y_train)
    seconds=perf_counter()-t0
    pred=np.asarray(model.predict(X_valid[selected])).astype(np.int8)
    tn,fp,fn,tp=map(int,confusion_matrix(labels,pred,labels=[0,1]).ravel())
    model_file=MODEL_DIR/f'lightgbm_{variant}_core16_v3.joblib'
    joblib.dump(model,model_file,compress=3)
    ablations.append({'variant':variant,'selected_features':len(selected),
       'accuracy_pct':100*accuracy_score(labels,pred),
       'attack_f1_pct':100*f1_score(labels,pred),
       'false_positive_pct':100*fp/max(1,tn+fp),
       'missed_attacks':fn,'false_alarms':fp,
       'training_seconds':seconds,'model_size_mib':model_file.stat().st_size/1048576})
    ablation_preds[variant]=pred
ablation_df=pd.DataFrame(ablations)
ablation_df.to_csv(METRICS/'edge_iiotset_v3_port_ablation.csv',index=False)
display(ablation_df.round(3))

## 6. Validate the ablation split assumption

Removing ports can make previously different Core16 rows identical. Measure any **train/validation no-port fingerprint overlap** before trusting the no-port results.

In [ ]:
# Hashes are a diagnostic screen, not a cryptographic proof of equivalence.
train_no=X_train[no_ports]
valid_no=X_valid[no_ports]
train_hash=set(pd.util.hash_pandas_object(train_no,index=False).to_numpy())
valid_hash=pd.util.hash_pandas_object(valid_no,index=False).to_numpy()
mask_overlap=np.isin(valid_hash,np.asarray(list(train_hash),dtype=np.uint64))
overlap_rows=int(mask_overlap.sum())
overlap_groups=int(pd.Series(valid_hash[mask_overlap]).nunique())
print('No-port validation records matching a training fingerprint:',overlap_rows)
print('Distinct matching validation fingerprints:',overlap_groups)
if overlap_rows:
    print('CAUTION: no-port ablation validation is not feature-group-disjoint.')
else:
    print('No matching no-port fingerprints detected between train and validation.')
subset_df=[]
unique_mask=~mask_overlap
for variant,pred in ablation_preds.items():
    if unique_mask.sum()>0 and len(np.unique(labels[unique_mask]))==2:
        tn,fp,fn,tp=map(int,confusion_matrix(labels[unique_mask],pred[unique_mask],labels=[0,1]).ravel())
        subset_df.append({'variant':variant,'subset':'nonoverlap_noport',
           'validation_rows':int(unique_mask.sum()),'f1_pct':100*f1_score(labels[unique_mask],pred[unique_mask]),
           'fpr_pct':100*fp/max(tn+fp,1),'fn':fn,'fp':fp})
    else:
        print('Unable to compute disjoint-only scores: subset absent or contains only one binary class.')
subset=pd.DataFrame(subset_df)
if not subset.empty: display(subset.round(3))
subset.to_csv(METRICS/'edge_iiotset_v3_ablation_disjoint_subset.csv',index=False)

## 7. Compare category recall and save graphs

Zero validation MITM records means MITM detection remains **unknown**, regardless of binary F1-score.

In [ ]:
by_type=[]
for variant,pred in ablation_preds.items():
    for typ in sorted(set(types)):
        ids=(types==typ)&(labels==1)
        n=int(ids.sum())
        if n==0: continue
        detections=int(np.sum(pred[ids]==1))
        by_type.append({'variant':variant,'attack_type':typ,'attack_records':n,
                        'detected':detections,'missed':n-detections,
                        'recall_pct':100*detections/n})
by_type_df=pd.DataFrame(by_type)
by_type_df.to_csv(METRICS/'edge_iiotset_v3_ablation_attack_types.csv',index=False)
display(by_type_df.sort_values(['attack_type','variant']).round(2))

fig,ax=plt.subplots(figsize=(8.2,4.8))
ax.scatter(metrics['false_positive_pct'],metrics['attack_f1_pct'])
for _,r in metrics.iterrows():
    ax.annotate(r['model'].replace('_',' ').title(),(r['false_positive_pct'],r['attack_f1_pct']),
                xytext=(4,4),textcoords='offset points',fontsize=9)
ax.set_title('Edge-IIoTset — Validation detection vs false alarms')
ax.set_xlabel('False-positive rate (%)');ax.set_ylabel('Attack F1-score (%)')
ax.grid(alpha=.2)
fig.tight_layout();fig.savefig(FIGURES/'edge_iiotset_v3_evaluation_tradeoff.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)

fig,axs=plt.subplots(1,2,figsize=(11.5,4.5))
name=ablation_df['variant'].str.replace('_',' ').str.title()
axs[0].bar(name,ablation_df['attack_f1_pct'])
axs[0].set_ylabel('Attack F1 (%)');axs[0].set_ylim(0,105)
axs[1].bar(name,ablation_df['false_positive_pct'])
axs[1].set_ylabel('False-positive rate (%)')
for ax in axs:
    ax.tick_params(axis='x',labelrotation=12);ax.grid(axis='y',alpha=.2)
fig.suptitle('Edge-IIoTset — LightGBM port reliance (validation only)')
fig.tight_layout();fig.savefig(FIGURES/'edge_iiotset_v3_port_ablation.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)

summary={'dataset':'Edge-IIoTset selected ML CSV','experiment':'Core16 V3 validation diagnostics',
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'source_rows':int(report['source_rows']),
    'train_rows':len(train),'validation_rows':len(valid),
    'frozen_manifest_sha256':report['manifest_sha256'],
    'attack_types_absent_from_validation':missing_types,
    'models_verified':MODEL_NAMES,
    'lightgbm_port_ablation':ablation_df.round(6).replace({np.nan:None}).to_dict(orient='records'),
    'core16_conflicting_label_training_rows':mixed_training_rows,
    'noport_overlap_with_train_validation_rows':overlap_rows,
    'noport_overlap_with_train_validation_fingerprint_groups':overlap_groups,
    'test_scored':False,
    'limitations':['No MITM validation records','Ports may be label shortcuts',
      'No-port data may have new cross-partition fingerprint collisions',
      'Group-disjoint features are not session-disjoint captures',
      'Laptop validation is not an edge gateway benchmark']}
(AUDITS/'edge_iiotset_v3_model_evaluation.json').write_text(
  json.dumps(summary,indent=2,allow_nan=False),encoding='utf-8')
print('Saved: 6 evaluation CSVs, 2 charts, JSON report and 2 diagnostic LightGBM models.')
print('TEST DATA NOT READ. Existing frozen splits unchanged.')